# 03 – Modelling
Goal: train and compare models that predict nightly price, starting from a simple baseline.

In [1]:
import numpy as np
import pandas as pd

clean = pd.read_csv("../data/processed/listings_clean.csv")
print(clean.shape)

(60852, 18)


In [2]:
from sklearn.model_selection import train_test_split

X = clean.drop(columns=["price_num", "log_price"])
y = clean["log_price"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Train:", len(X_train), " Validation:", len(X_val), " Test:", len(X_test))

Train: 42596  Validation: 9128  Test: 9128


In [3]:
from sklearn.metrics import mean_absolute_error, r2_score

def evaluate(name, y_true_log, y_pred_log):
    true_price = np.exp(y_true_log)
    pred_price = np.exp(y_pred_log)
    return {
        "Model": name,
        "MAE (£)": round(mean_absolute_error(true_price, pred_price), 1),
        "Median % error": round(np.median(np.abs(pred_price - true_price) / true_price) * 100, 1),
        "R² (log)": round(r2_score(y_true_log, y_pred_log), 3),
    }

results = []

In [4]:
baseline_pred = np.full(len(y_val), y_train.median())
results.append(evaluate("Baseline (median price)", y_val, baseline_pred))
pd.DataFrame(results)

,Model,MAE (£),Median % error,R² (log)
0,Baseline (median price),120.8,48.8,-0.002


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ["neighbourhood_cleansed", "room_type"]
num_cols = [c for c in X.columns if c not in cat_cols]

def make_preprocessor(scale):
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

models = {
    "Linear regression": Pipeline([
        ("prep", make_preprocessor(scale=True)),
        ("model", LinearRegression()),
    ]),
    "Random forest": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("model", RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42)),
    ]),
    "Gradient boosting": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("model", HistGradientBoostingRegressor(random_state=42)),
    ]),
}

In [7]:
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    results.append(evaluate(name, y_val, pipe.predict(X_val)))
    print("Done:", name)

Done: Linear regression
Done: Random forest
Done: Gradient boosting


In [8]:
results_df = pd.DataFrame(results)
results_df

,Model,MAE (£),Median % error,R² (log)
0,Baseline (median price),120.8,48.8,-0.002
1,Linear regression,68.1,23.8,0.676
2,Random forest,56.5,19.6,0.755
3,Gradient boosting,58.5,21.1,0.746


In [9]:
settings = [
    {"learning_rate": 0.1, "max_iter": 300, "max_leaf_nodes": 31},
    {"learning_rate": 0.05, "max_iter": 600, "max_leaf_nodes": 31},
    {"learning_rate": 0.05, "max_iter": 600, "max_leaf_nodes": 63},
    {"learning_rate": 0.05, "max_iter": 1000, "max_leaf_nodes": 63, "min_samples_leaf": 40},
]

tuning = []
for s in settings:
    pipe = Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("model", HistGradientBoostingRegressor(random_state=42, **s)),
    ])
    pipe.fit(X_train, y_train)
    tuning.append(evaluate(str(s), y_val, pipe.predict(X_val)))
    print("Done:", s)

pd.DataFrame(tuning)

Done: {'learning_rate': 0.1, 'max_iter': 300, 'max_leaf_nodes': 31}
Done: {'learning_rate': 0.05, 'max_iter': 600, 'max_leaf_nodes': 31}
Done: {'learning_rate': 0.05, 'max_iter': 600, 'max_leaf_nodes': 63}
Done: {'learning_rate': 0.05, 'max_iter': 1000, 'max_leaf_nodes': 63, 'min_samples_leaf': 40}


,Model,MAE (£),Median % error,R² (log)
0,"{'learning_rate': 0.1, 'max_iter': 300, 'max_l...",57.7,20.5,0.752
1,"{'learning_rate': 0.05, 'max_iter': 600, 'max_...",57.7,20.5,0.753
2,"{'learning_rate': 0.05, 'max_iter': 600, 'max_...",57.3,20.4,0.754
3,"{'learning_rate': 0.05, 'max_iter': 1000, 'max...",57.2,20.2,0.756


In [10]:
import os
import joblib

joblib.dump(models["Random forest"], "../models/rf_size_check.joblib")
size_mb = os.path.getsize("../models/rf_size_check.joblib") / 1_000_000
print(f"Random forest file size: {size_mb:.0f} MB")
os.remove("../models/rf_size_check.joblib")

Random forest file size: 337 MB


In [11]:
best_params = {"learning_rate": 0.05, "max_iter": 1000, "max_leaf_nodes": 63, "min_samples_leaf": 40}

final_model = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("model", HistGradientBoostingRegressor(random_state=42, **best_params)),
])

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)

test_result = evaluate("Final model (test set)", y_test, final_model.predict(X_test))
pd.DataFrame([test_result])

,Model,MAE (£),Median % error,R² (log)
0,Final model (test set),57.0,20.2,0.762


In [12]:
joblib.dump(final_model, "../models/price_model.joblib")
size_mb = os.path.getsize("../models/price_model.joblib") / 1_000_000
print(f"Saved final model: {size_mb:.1f} MB")

Saved final model: 2.2 MB


## Approach
- Data split into training (70%), validation (15%) and test (15%) sets. Models are compared on the validation set; the test set is used once at the end for the final score.
- Missing values are filled and categories encoded inside a pipeline, so these steps learn only from the training data, avoiding data leakage.
- Three models were compared against a baseline that predicts the median price for every listing.
## Validation results
| Model | MAE (£) | Median % error | R² (log) |
|---|---|---|---|
| Baseline (median price) | 120.8 | 48.8 | 0.00 |
| Linear regression | 68.1 | 23.8 | 0.68 |
| Random forest | 56.5 | 19.6 | 0.76 |
| Gradient boosting (default settings) | 58.5 | 21.1 | 0.75 |

All three models beat the baseline, more than halving its error. Tree-based models outperform linear regression, likely because price depends on interactions between features (for example, the value of an extra bedroom varies by location).

## Choosing the final model
Gradient boosting was tuned by trying four combinations of settings. The best (learning rate 0.05, up to 1,000 trees, 63 leaves per tree, at least 40 listings per leaf) reached a validation MAE of £57.2, a median error of 20.2% and an R² of 0.756.

This is practically identical to the random forest (MAE £56.5, median error 19.6%, R² 0.755), but the random forest's saved file is 337 MB, too large for GitHub and slow to load in a web app. Tuned gradient boosting was therefore chosen as the final model.

The final model was retrained on the training and validation sets combined, then evaluated once on the untouched test set.


## Final test result
| Model | MAE (£) | Median % error | R² (log) |
|---|---|---|---|
| Tuned gradient boosting (test set) | 57.0 | 20.2 | 0.762 |

The test results closely match the validation results, showing that the model generalises well to listings it has never seen. The saved model is 2.2 MB, about 150 times smaller than the random forest.